<a href="https://colab.research.google.com/github/carbon9634-max/nasa-hotspot-explorer/blob/main/Submission_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import pathlib
import requests


BASE = pathlib.Path('/content/drive/MyDrive/Wildfire Data/brazil')
RAW, OUT = BASE/'raw', BASE/'processed'
RAW.mkdir(parents = True, exist_ok = True); OUT.mkdir(exist_ok = True)

Mounts google drive and imports the pibrarys. Defines folders raw/ for dounloads and processed/ for results.
                                                 ^ |

Produces variables RAW and OUT for folder paths. |

In [ ]:
SOURCES = {
  'MODIS': ('https://firms.modaps.eosdis.nasa.gov/data/country/modis/{y}/modis_{y}_Brazil.csv', range(2001, 2027)),
  'VIIRS': ('https://firms.modaps.eosdis.nasa.gov/data/country/viirs-snpp/{y}/viirs-snpp_{y}_Brazil.csv', range(2012, 2027)),
}
for sensor, (pattern, years) in SOURCES.items():
    for y in years:
        f = RAW / f'{sensor}_{y}.csv'
        if f.exists(): continue
        r = requests.get(pattern.format(y=y), timeout=300)
        print(sensor, y, r.status_code, len(r.content))
        if r.ok: f.write_bytes(r.content)

In [ ]:
for f in sorted(RAW.glob('*.csv')):
    print(f.name, round(f.stat().st_size / 1e6, 1), 'MB')

In [ ]:
m = pd.read_csv(RAW/'MODIS_2015.csv', nrows=5)
v = pd.read_csv(RAW/'VIIRS_2015.csv', nrows=5)
print(m.columns.tolist())
print(v.columns.tolist())
print(m['confidence'].tolist())
print(v['confidence'].tolist())

Downloads one CSV per year from FIRMS: MODIS 2001 onward and VIIRS S-NPP 2012 onward. Files that already exist are skipped.
Produces about 29 csv files in raw/

In [ ]:
def normalize(df, sensor):
    t = df['acq_time'].astype(str).str.zfill(4)
    out = pd.DataFrame({
        'sensor': sensor,
        'satellite': df['satellite'],
        'date': pd.to_datetime(df['acq_date']),
        'hhmm': t,
        'lat': df['latitude'], 'lon': df['longitude'],
        'frp': pd.to_numeric(df['frp'], errors='coerce'),
        'daynight': df['daynight'],
        'type': pd.to_numeric(df['type'], errors='coerce').fillna(0) if 'type' in df else 0,  # NEW
    })
    c = df['confidence'].astype(str).str.strip().str.lower()
    num = pd.to_numeric(c, errors='coerce')
    cls = pd.cut(num, [-1, 29, 79, 100], labels=['low', 'nominal', 'high']).astype(object)
    letters = c.map({'l': 'low', 'n': 'nominal', 'h': 'high'})
    out['conf_class'] = cls.where(num.notna(), letters)
    return out

The normalize function puts both sensors in one format, Defines a function that converts raw FIRMS table into a common dataset. It also maps confidence to one scale. Produces nothing by itself.

In [ ]:
def aggregate(df, grid=0.25):
    df = df[df['conf_class'].isin(['nominal', 'high']) & (df['type'] == 0)].copy()  # NEW
    df['k_lat'] = (df['lat'] / 0.01).round().astype('int32')
    df['k_lon'] = (df['lon'] / 0.01).round().astype('int32')
    df = df.drop_duplicates(['sensor', 'date', 'k_lat', 'k_lon'])
    df['cell_lat'] = (np.floor(df['lat'] / grid) * grid + grid/2).round(4)
    df['cell_lon'] = (np.floor(df['lon'] / grid) * grid + grid/2).round(4)
    return (df.groupby(['sensor', 'date', 'cell_lat', 'cell_lon'])
              .agg(n_hotspots=('frp', 'size'), frp_sum=('frp', 'sum'))
              .reset_index())

parts = []
for f in sorted(RAW.glob('*.csv')):
    sensor = f.name.split('_')[0]
    df = pd.read_csv(f, usecols=lambda c: c in
        ['latitude','longitude','acq_date','acq_time','satellite','confidence','frp','daynight','type'])  # NEW: added 'type'
    parts.append(aggregate(normalize(df, sensor)))
    print(f.name, len(df))

daily = pd.concat(parts, ignore_index=True)
daily.to_parquet(OUT/'brazil_daily_grid.parquet', index=False)
print(daily.shape, daily['date'].min(), daily['date'].max())

Aggregate is a proccessing loop. It keeps only nominal and high confidence and type == 0(vegetation fires) so it excludes natural gases, petrol and stuff like that.
Snaps detections to a 1km box and keeps one per box per day per day. So a 375m VIIRS count means a km square had a fire, which is comparable to MODIS.
Counts hotpots per 0.25 degree grid square per day so it can be inputed in a interactive map.

In [ ]:
monthly = (daily.assign(ym=daily['date'].dt.to_period('M').dt.to_timestamp())
                .groupby(['ym', 'sensor'])['n_hotspots'].sum().unstack())
ax = monthly.loc['2012':].plot(figsize=(12, 4), title='Monthly hotspots, Brazil')

Montly hotspots chart, shows the difference between the two sensors. It adds up hotspots per month for each sensor and plots MODIS against VIIRS from 2012 on. Both lines have the same seasonal shape with VIIRS higher which means that both sensors detected fires in the same relative area but VIIRS detected more, which isnt suprising since its more accurate.

In [ ]:
yearly = (daily.assign(year=daily['date'].dt.year)
               .groupby(['year', 'sensor'])['n_hotspots'].sum().unstack())
yearly['ratio'] = yearly['VIIRS'] / yearly['MODIS']
print(yearly.loc[2012:].round(2))

Adds up hotspots per year per sensor and divides VIIRS by MODIS. This is to see on average how many more detections there are in VIIRS compared to MODIS.

In [ ]:
monthly = (daily.assign(ym=daily['date'].dt.to_period('M').dt.to_timestamp())
                .groupby(['ym', 'sensor'])['n_hotspots'].sum().unstack())
overlap = monthly.loc['2012-02':'2025-12'].dropna().copy()
k = overlap['VIIRS'].sum() / overlap['MODIS'].sum()
overlap['VIIRS_adj'] = overlap['VIIRS'] / k
r = overlap['MODIS'].corr(overlap['VIIRS'])
err_before = (overlap['VIIRS'] - overlap['MODIS']).abs().mean() / overlap['MODIS'].mean()
err_after = (overlap['VIIRS_adj'] - overlap['MODIS']).abs().mean() / overlap['MODIS'].mean()
print('k =', round(k, 3), '| correlation =', round(r, 3))
print('average error before:', round(100 * err_before), '%  after:', round(100 * err_after), '%')
overlap[['MODIS', 'VIIRS_adj']].plot(figsize=(12, 4), title='After calibration')

FOr the monghts when sensors flew (Feb 12 to Dec 2025), it computes k which is the total VIIRS/ total MODIS and creaters a VIIRS_adk = VIIRS/k. It lists the correlation and the average error before and after adjusting. You can see how both the datasets are on the same scale now.

In [ ]:
train = overlap.loc[:'2018-12']
test = overlap.loc['2019-01':]
k_train = train['VIIRS'].sum() / train['MODIS'].sum()
pred = test['VIIRS'] / k_train
err = (pred - test['MODIS']).abs().mean() / test['MODIS'].mean()
print('k from 2012-2018 =', round(k_train, 3), '| error on 2019-2025:', round(100 * err), '%')

This is a machine learning thing where you just see if the this data holds up in the later years. Its split into a train and a test so the training data can be used to see if the held out test data is the same.

In [ ]:
START = pd.Timestamp('2012-02-01')
d = daily.copy()
d['ym'] = d['date'].dt.to_period('M').dt.to_timestamp()
keep = ((d['sensor'] == 'MODIS') & (d['date'] < START)) | ((d['sensor'] == 'VIIRS') & (d['date'] >= START))
d = d[keep]
d['harmonized'] = np.where(d['sensor'] == 'VIIRS', d['n_hotspots'] / k, d['n_hotspots'])
cells = (d.groupby(['ym', 'cell_lat', 'cell_lon'])
           .agg(harmonized=('harmonized', 'sum'), raw=('n_hotspots', 'sum'))
           .reset_index())
cells['harmonized'] = cells['harmonized'].round(2)
cells.to_parquet(OUT/'brazil_monthly_cells.parquet', index=False)
print(cells.shape)
series = cells.groupby('ym')['harmonized'].sum()
series.plot(figsize=(12, 4), title='Harmonized monthly hotspots, Brazil (MODIS-equivalent)')

This is what splices the datasets into one harmoized series. Uses one sensor at a time to avoid double counting. ODIS before 2012-02-01, and VIIRS divided by k from 2012-02-01 on. Adds up counts per month per grid square and saves the result.

In [ ]:
def calendar_table(cells, lat_min, lat_max, lon_min, lon_max):
    sub = cells[(cells['cell_lat'].between(lat_min, lat_max)) &
                (cells['cell_lon'].between(lon_min, lon_max))]
    sub = sub.assign(year=sub['ym'].dt.year, month=sub['ym'].dt.month)
    return sub.pivot_table(index='year', columns='month', values='harmonized', aggfunc='sum')

tbl = calendar_table(cells, -18, -8, -62, -50)

Just makes a function that take latitude and longtitude, returns the year by year month table of harmized hotspot totals for that area. The example it runs for creates a box around mato grosso.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(tbl.values, aspect='auto', cmap='cividis')
ax.set_xticks(range(12))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_yticks(range(len(tbl.index)))
ax.set_yticklabels(tbl.index)
fig.colorbar(im, label='Harmonized hotspots')
ax.set_title('Burn calendar: Mato Grosso (approx.)')
plt.show()

In [ ]:
Draw the calendar as a year-by-month heapmap using cividis colormap.

In [ ]:
import json, os
summary = {'k': round(float(k), 3), 'correlation': round(float(r), 3),
           'error_before_pct': round(float(100 * err_before)), 'error_after_pct': round(float(100 * err_after)),
           'splice_date': '2012-02-01', 'confidence_filter': 'nominal+high', 'grid_deg': 0.25}
(OUT/'harmonization_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)
print(round(os.path.getsize(OUT/'brazil_monthly_cells.parquet') / 1e6, 1), 'MB')

In [ ]:
summary['holdout_k'] = round(float(k_train), 3)
summary['holdout_error_pct'] = round(float(100 * err))
(OUT/'harmonization_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)

In [ ]:
from google.colab import userdata
MAP_KEY = userdata.get('MAP_KEY')
url = f'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{MAP_KEY}/ALL'
avail = pd.read_csv(url)
print(avail.to_string())

Takes a map key and loads real time global data to scale the brazil code up to the whole world. Only harmozied one day of data however.

In [ ]:
import geopandas as gpd
modis_g = pd.read_csv('/content/drive/MyDrive/Wildfire Data/fire_nrt_M-C61_835294.csv')
viirs_g = pd.read_csv('/content/drive/MyDrive/Wildfire Data/fire_nrt_SV-C2_835297.csv')
g = pd.concat([aggregate(normalize(modis_g, 'MODIS')),
               aggregate(normalize(viirs_g, 'VIIRS'))], ignore_index=True)

url = 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson'
world = gpd.read_file(url)[['CONTINENT', 'geometry']]
pts = g[['cell_lat', 'cell_lon']].drop_duplicates()
pts = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts['cell_lon'], pts['cell_lat']), crs='EPSG:4326')
pts = gpd.sjoin(pts, world, how='left', predicate='within')
pts = pts[['cell_lat', 'cell_lon', 'CONTINENT']].drop_duplicates(['cell_lat', 'cell_lon'])
g = g.merge(pts, on=['cell_lat', 'cell_lon'], how='left')

tot = g.groupby(['CONTINENT', 'sensor'])['n_hotspots'].sum().unstack()
tot['ratio'] = tot['VIIRS'] / tot['MODIS']
print(tot.round(2))

Loads global september 2026 MODIS and VIIRS files. Cleans and grids them with the same normalize and aggregate.

In [ ]:
import matplotlib.pyplot as plt

DAY = pd.Timestamp('2026-09-15')
k_cont = tot['ratio'].drop('Antarctica').to_dict()
k_all = tot['VIIRS'].sum() / tot['MODIS'].sum()

day = g[g['date'] == DAY].copy()
day['k'] = day['CONTINENT'].map(k_cont).fillna(k_all)

piv = day.pivot_table(index='CONTINENT', columns='sensor', values='n_hotspots', aggfunc='sum')
piv = piv.drop('Antarctica', errors='ignore')
piv['k_continent'] = piv.index.map(k_cont)
piv['VIIRS_adj_continent'] = piv['VIIRS'] / piv['k_continent']
piv['VIIRS_adj_global'] = piv['VIIRS'] / k_all
piv['err_continent_pct'] = (piv['VIIRS_adj_continent'] - piv['MODIS']).abs() / piv['MODIS'] * 100
piv['err_global_pct'] = (piv['VIIRS_adj_global'] - piv['MODIS']).abs() / piv['MODIS'] * 100
print(piv.round(1))
print(piv[['err_continent_pct', 'err_global_pct']].mean().round(1))
snap = day[day['sensor'] == 'VIIRS'].copy()
snap['harmonized'] = (snap['n_hotspots'] / snap['k']).round(2)
snap[['cell_lat', 'cell_lon', 'CONTINENT', 'harmonized']].to_csv(OUT/'world_2026-09-15_harmonized.csv', index=False)


snap = snap.sort_values('harmonized')
fig, ax = plt.subplots(figsize=(14, 6))
world.boundary.plot(ax=ax, color='gray', linewidth=0.4)
sc = ax.scatter(snap['cell_lon'], snap['cell_lat'], c=np.log10(1 + snap['harmonized']),
                s=10, cmap='viridis', vmin=0, vmax=np.log10(1 + snap['harmonized'].quantile(0.99)))
ax.set_xlim(-180, 180)
ax.set_ylim(-60, 80)
fig.colorbar(sc, label='log10(1 + harmonized hotspots), clipped at the 99th percentile')
ax.set_title('Harmonized hotspots, whole world, 2026-09-15 (MODIS-equivalent)')
plt.show()

Basically same code for brazil but scales up to the whole world.

In [ ]:
piv.round(2).to_csv(OUT/'world_2026-09-15_check.csv')

Just saves the file.

Limitations: The viirs /modis ratio drifts over time so a single k can slightly underestimate recent peak months.

Terra flew in 2001(Aqua joined in 2002) so 2001 counts may be low

MOdis is the reference scale, Errors measure how well VIIRS matches to modis, not how many real fire occured
The world check is one month and one day, per region calibration with multi year is later

Brazil archive ends in december 2025.